In [ ]:
import os
import sys
import glob
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from tensorflow.keras import layers, models
from PIL import Image
from tqdm import tqdm
from dotenv import load_dotenv
from pathlib import Path
from torchvision import transforms
from torch.utils.data import Subset
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import callbacks
from tqdm.keras import TqdmCallback

sys.path.append(os.path.abspath(os.path.join("../..")))
from src.datasets.huggingfacedataset import HuggingFaceDataset

load_dotenv()

PROJECT_ROOT = Path(os.environ["PROJECT_ROOT_DIR"])
os.chdir(PROJECT_ROOT)
captcha_dir = Path(PROJECT_ROOT) / "data" / "hammer_captchas"

In [11]:
file_paths = glob.glob(str(captcha_dir / "*.jpg"))[:10000] # Or .jpg
path_ds = tf.data.Dataset.from_tensor_slices(file_paths)

def load_and_preprocess(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=1) # Decodes & Grayscale in one go
    img = tf.image.resize(img, [40, 150])
    img = tf.cast(img, tf.float32) / 255.0 # Normalization [0, 1]
    return img, img # (Input, Target) for Autoencoder

# 3. Build the parallel pipeline
dataset = tf.data.Dataset.from_tensor_slices(file_paths)
dataset = dataset.shuffle(buffer_size=1024)

# Use num_parallel_calls to use all CPU cores
dataset = dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)

# Batch and Prefetch (This is the "Secret Sauce" for speed)
train_ds = dataset.skip(1000).batch(64).prefetch(tf.data.AUTOTUNE)
val_ds = dataset.take(1000).batch(64).prefetch(tf.data.AUTOTUNE)

In [26]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_anomaly_autoencoder_bce():
    input_img = layers.Input(shape=(40, 150, 1)) 

    # --- ENCODER ---
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(input_img)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2), padding='same')(x)
    
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2), padding='same')(x)

    # --- BOTTLENECK ---
    # REDUCED from 32 to 16. This makes it harder for the model to 
    # generalize to non-captcha images.
    bottleneck = layers.Conv2D(16, (3, 3), activation='relu', padding='same', name="latent_space")(x)

    # --- DECODER ---
    x = layers.Conv2DTranspose(64, (3, 3), strides=(2, 2), padding='same')(bottleneck)
    x = layers.Activation('relu')(x)
    x = layers.BatchNormalization()(x)

    x = layers.Conv2DTranspose(32, (3, 3), strides=(2, 2), padding='same')(x)
    x = layers.Activation('relu')(x)
    
    x = layers.Cropping2D(cropping=((0, 0), (1, 1)))(x)

    # Output Layer - Sigmoid is REQUIRED for Binary Cross Entropy
    decoded = layers.Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)

    return models.Model(input_img, decoded)

autoencoder = build_anomaly_autoencoder_bce()

# Using 'binary_crossentropy' as a string is the standard way
autoencoder.compile(
    optimizer='adam',
    loss='binary_crossentropy'
)

autoencoder.summary()

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 40, 150, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_38 (Conv2D)              │ (None, 40, 150, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_29          │ (None, 40, 150, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 20, 75, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_39 (Conv2D)              │ (None, 20, 75, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_30          │ (None, 20, 75, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ (None, 10, 38, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ latent_space (Conv2D)           │ (None, 10, 38, 16)     │         9,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_14             │ (None, 20, 76, 64)     │         9,280 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 20, 76, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_31          │ (None, 20, 76, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_15             │ (None, 40, 152, 32)    │        18,464 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_7 (Activation)       │ (None, 40, 152, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cropping2d_6 (Cropping2D)       │ (None, 40, 150, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_40 (Conv2D)              │ (None, 40, 150, 1)     │           289 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 56,721 (221.57 KB)

 Trainable params: 56,401 (220.32 KB)

 Non-trainable params: 320 (1.25 KB)

In [27]:
# Early stopping to prevent unnecessary epochs
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

steps_per_epoch = len(train_idx) // batch_size
validation_steps = len(val_idx) // batch_size

history = autoencoder.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=[early_stop]
)

Epoch 1/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - loss: 0.2624 - val_loss: 0.4892
Epoch 2/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 10s 27ms/step - loss: 0.2010 - val_loss: 0.4919
Epoch 3/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 0.2168 - val_loss: 0.3962
Epoch 4/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1893 - val_loss: 0.3898
Epoch 5/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 0.2106 - val_loss: 0.3043
Epoch 6/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2266 - val_loss: 0.3051
Epoch 7/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 0.2075 - val_loss: 0.2187
Epoch 8/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1795 - val_loss: 0.2224
Epoch 9/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.2047 - val_loss: 0.2100
Epoch 10/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1929 - val_loss: 0.2059
Epoch 11/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 0.2038 - val_loss: 0.2070
Epoch 12/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s

In [28]:
import os
from pathlib import Path

save_dir = Path(PROJECT_ROOT) / "src" / "models" / "autoencoder"
os.makedirs(save_dir, exist_ok=True)

model_path = save_dir / "autoencoder_v1.keras"

# Save model
autoencoder.save(model_path)